# Semana 5: Ridge, Lasso y selección por validación

**Pregunta de trabajo.** Verificar la convención `alpha = m * lambda`, trayectorias de coeficientes y selección sin tocar test.

In [1]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

In [2]:
data = load_diabetes()
X_train, X_temp, y_train, y_temp = train_test_split(data.data, data.target, test_size=0.4, random_state=2105)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=2105)

In [3]:
for alpha in [0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    print(alpha, mean_squared_error(y_val, model.predict(X_val)))

0.01 2774.0295333400377
0.1 2775.027022979077
1 2783.345233236807
10 2813.7060897214715
100 2994.216503554033


## Convenciones y trayectorias

Se traduce la penalización promedio del curso a `Ridge(alpha=m*lambda)` y se compara Lasso sin usar test para elegir.

In [4]:
import numpy as np
import pandas as pd

alphas = [0.01, 0.1, 1.0, 10.0, 100.0]
ridge_rows = []
ridge_models = {}
for alpha in alphas:
    candidate = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    candidate.fit(X_train, y_train)
    ridge_rows.append({"alpha": alpha, "mse_dev": mean_squared_error(y_val, candidate.predict(X_val))})
    ridge_models[alpha] = candidate
ridge_results = pd.DataFrame(ridge_rows)
ridge_results

,alpha,mse_dev
0,0.01,2774.029533
1,0.10,2775.027023
2,1.00,2783.345233
3,10.00,2813.706090
4,100.00,2994.216504


In [5]:
best_alpha = float(ridge_results.loc[ridge_results["mse_dev"].idxmin(), "alpha"])
scaler = ridge_models[best_alpha].named_steps["standardscaler"]
Xz = scaler.transform(X_train)
y_mean = y_train.mean()
coef_manual = np.linalg.solve(Xz.T @ Xz + best_alpha * np.eye(Xz.shape[1]), Xz.T @ (y_train - y_mean))
coef_library = ridge_models[best_alpha].named_steps["ridge"].coef_
{
    "alpha": best_alpha,
    "lambda_curso": best_alpha / len(y_train),
    "error_coeficientes": float(np.linalg.norm(coef_manual - coef_library)),
}

{'alpha': 0.01,
 'lambda_curso': 3.7735849056603776e-05,
 'error_coeficientes': 2.292744212268888e-13}

In [6]:
lasso_rows = []
for alpha in np.logspace(-3, 1, 10):
    candidate = make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=30000))
    candidate.fit(X_train, y_train)
    coef = candidate.named_steps["lasso"].coef_
    lasso_rows.append({
        "alpha": alpha,
        "mse_dev": mean_squared_error(y_val, candidate.predict(X_val)),
        "activos": int(np.count_nonzero(coef)),
    })
lasso_results = pd.DataFrame(lasso_rows)
lasso_results

,alpha,mse_dev,activos
0,0.001000,2773.975362,10
1,0.002783,2774.081046,10
2,0.007743,2774.386501,10
3,0.021544,2775.325123,10
4,0.059948,2778.622357,10
5,0.166810,2787.553098,9
6,0.464159,2788.871472,9
7,1.291550,2807.712528,8
8,3.593814,2919.863642,6
9,10.000000,3094.119595,4


In [7]:
best_lasso_alpha = float(lasso_results.loc[lasso_results["mse_dev"].idxmin(), "alpha"])
final_lasso = make_pipeline(StandardScaler(), Lasso(alpha=best_lasso_alpha, max_iter=30000))
final_lasso.fit(X_train, y_train)
test_summary = {
    "ridge_alpha": best_alpha,
    "ridge_mse_test": mean_squared_error(y_test, ridge_models[best_alpha].predict(X_test)),
    "lasso_alpha": best_lasso_alpha,
    "lasso_mse_test": mean_squared_error(y_test, final_lasso.predict(X_test)),
}
test_summary

{'ridge_alpha': 0.01,
 'ridge_mse_test': 3006.9750631592033,
 'lasso_alpha': 0.001,
 'lasso_mse_test': 3006.984234356316}

In [8]:
assert np.linalg.norm(coef_manual - coef_library) < 1e-8
print("El test se consultó después de fijar alpha con dev; un coeficiente cero no prueba irrelevancia poblacional.")

El test se consultó después de fijar alpha con dev; un coeficiente cero no prueba irrelevancia poblacional.
